# NFA to DFA Conversion

This notebook introduces the idea of converting a nondeterministic finite automaton (NFA) into an equivalent deterministic finite automaton (DFA). The goal is to understand why this conversion is useful and how the subset construction works in a clear, step-by-step way.

We will study the core theory, see small examples, and then implement the conversion in Python.


## 👨‍💻 Author

### **Muhammad Ali**

[![GitHub](https://img.shields.io/badge/GitHub-AliMuhammad78-181717?style=flat-square\&logo=github\&logoColor=white)](https://github.com/AliMuhammad78)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Muhammad%20Ali-0A66C2?style=flat-square\&logo=linkedin\&logoColor=white)](https://www.linkedin.com/in/muhammad-ali-91294a290)
[![Kaggle](https://img.shields.io/badge/Kaggle-ali98muhammad45-20BEFF?style=flat-square\&logo=kaggle\&logoColor=white)](https://www.kaggle.com/ali98muhammad45)

## Learning objectives

By the end of this notebook, you should be able to:

- explain the difference between an NFA and a DFA,
- describe the meaning of epsilon transitions and nondeterminism,
- understand why every NFA can be converted into an equivalent DFA,
- apply the subset construction algorithm on small examples,
- implement the conversion in Python and test it on example automata.

## Basic theory

A finite automaton is a mathematical model for recognizing strings. It reads symbols one by one and decides whether the input is accepted or rejected.

A DFA has a very important property: for each state and each input symbol, there is exactly one possible next state. This makes its behavior deterministic.

An NFA is more flexible: from a state and an input symbol, there may be several possible next states. Also, an NFA may have epsilon transitions, which allow it to move without consuming any input symbol.

Although NFAs look more general, they are not more powerful than DFAs. In fact, every NFA has an equivalent DFA that recognizes the same language.

The key idea is to represent the set of all possible states the automaton could be in after reading a prefix of the input. In a DFA, each state corresponds to a subset of NFA states. This is the subset construction.

## Definitions

Let an NFA be defined by:

- Q: a finite set of states,
- Σ: an input alphabet,
- δ: the transition function (or transition relation),
- q0: the start state,
- F: the set of accepting states.

For a DFA, the transition function is usually written as:

δ_D : Q_D × Σ → Q_D

For an NFA, the transition may be multivalued:

δ_N : Q × Σ → P(Q)

where P(Q) is the power set of Q, meaning the set of all subsets of Q.

When converting an NFA to a DFA, each DFA state is a subset of NFA states. If the NFA can be in any state from a set S after reading some input, then the DFA will be in the corresponding state S.

## Epsilon closure

The epsilon closure of a state is the set of all states reachable from it using only epsilon transitions (zero or more epsilon moves). This matters because even if the NFA does not consume input, it may still move to other states.

For example, if q1 has an epsilon transition to q2 and q2 has an epsilon transition to q3, then:

epsilon_closure(q1) = {q1, q2, q3}

In the subset construction, we always begin from the epsilon closure of the start state.

## Important terminology and notation

- Alphabet: a finite set of input symbols, often written as Σ.
- String: a sequence of symbols from Σ.
- Language: a set of strings accepted by the automaton.
- State: a node representing a machine configuration.
- Start state: the state where processing begins.
- Accepting state: a state that causes the input to be accepted if reached after the final symbol.
- Transition: a move from one state to another on an input symbol.
- Nondeterminism: multiple possible moves exist for the same symbol.
- Determinism: exactly one move exists for each state-symbol pair.
- Subset construction: the algorithm used to convert an NFA into an equivalent DFA.

## Why the conversion works

The DFA tracks all possible NFA states after reading each prefix of the input. This is enough because the language of an NFA depends on whether at least one possible path leads to an accepting state.

If the NFA can reach accepting state(s) along some path, then the corresponding DFA state is accepting. If every path rejects, the DFA state is nonaccepting.

This gives us an equivalent automaton that is deterministic but recognizes the same language.

## Step-by-step subset construction

Given an NFA:

1. Compute epsilon-closure of the start state.
2. Treat that closure as a DFA state.
3. For each symbol in the alphabet, compute the set of NFA states reachable by moving on that symbol from each state in the current subset.
4. Take the epsilon closure of each result.
5. If this new subset has not been seen before, add it as a new DFA state.
6. Repeat until no new subsets are created.
7. Mark any DFA state containing at least one accepting NFA state as accepting.

That is the full algorithm.

## Simple example: NFA with two possible paths

Suppose we have an NFA with states {q0, q1, q2}, alphabet {a, b}, start state q0, and accepting state q2. There are transitions:

- q0 --a--> q0
- q0 --a--> q1
- q1 --b--> q2

This NFA accepts strings that contain an a followed eventually by a b, with some extra a's before the b.

The DFA would track sets like:

- {q0}
- {q0, q1}
- {q0, q1, q2}

These subsets correspond to different stages of reading the input.

## Python code: representing the NFA

The following code defines a very small NFA as a dictionary of transitions. We will use the subset construction to build a DFA.


In [2]:
# Example NFA representation

# States: q0, q1, q2
# Alphabet: {'a', 'b'}
# Start state: q0
# Accepting state: q2

nfa = {
    'q0': {'a': {'q0', 'q1'}},
    'q1': {'b': {'q2'}},
    'q2': {}
}

# Print the transition structure
for state in nfa:
    print(f"{state}: {nfa[state]}")

print("\nThis NFA can be in multiple states after reading the same symbol.")


q0: {'a': {'q1', 'q0'}}
q1: {'b': {'q2'}}
q2: {}

This NFA can be in multiple states after reading the same symbol.


## Example 1: reachability by subset tracking

A useful way to understand the conversion is to track all possible states after each input. Suppose the NFA is reading the string `aaab`.

- Initially, the NFA is in {q0}
- After reading `a`, it may be in {q0, q1}
- After another `a`, it may still be in {q0, q1}
- After the final `b`, it may reach {q2}

If {q2} is accepting, then the string is accepted.

This tells us that the DFA should have a state representing the subset {q0, q1} and another representing {q2}.

## Example 2: avoiding duplicate states

When converting, we do not want to create unnecessary DFA states. If a subset of NFA states has already been seen, we reuse the same DFA state instead of creating a duplicate.

This is important because the finite automaton must remain finite. The number of possible subsets is finite, although it can grow exponentially in the worst case.

## Python implementation: subset construction

Below is a beginner-friendly implementation of the subset construction. The code follows the standard algorithm closely.


In [3]:
def epsilon_closure(states, epsilon_transitions):
    """Return all states reachable from the given set using only epsilon moves."""
    closure = set(states)
    stack = list(states)

    while stack:
        current = stack.pop()
        for nxt in epsilon_transitions.get(current, set()):
            if nxt not in closure:
                closure.add(nxt)
                stack.append(nxt)

    return closure


def nfa_to_dfa(nfa, alphabet):
    """
    Convert an NFA to an equivalent DFA using the subset construction.

    nfa is a dictionary of the form:
        {
            'q0': {'a': {'q0','q1'}, 'b': set()},
            'q1': {'b': {'q2'}},
            'q2': {}
        }

    epsilon_transitions is a dictionary of the form:
        {'q0': {'q1', 'q2'}}
    """
    epsilon_transitions = nfa.get('epsilon', {})
    start_state = frozenset(epsilon_closure({'q0'}, epsilon_transitions))

    dfa_states = {}
    queue = [start_state]
    visited = set()

    while queue:
        current = queue.pop(0)
        if current in visited:
            continue
        visited.add(current)

        dfa_states[current] = {}

        for symbol in alphabet:
            next_states = set()
            for state in current:
                for nxt in nfa.get(state, {}).get(symbol, set()):
                    next_states.add(nxt)

            # Apply epsilon closure after each symbol transition.
            next_subset = frozenset(epsilon_closure(next_states, epsilon_transitions))
            dfa_states[current][symbol] = next_subset

            if next_subset not in visited and next_subset not in queue:
                queue.append(next_subset)

    return dfa_states, start_state


# Example NFA from the theory section
nfa = {
    'q0': {'a': {'q0', 'q1'}},
    'q1': {'b': {'q2'}},
    'q2': {},
    'epsilon': {}
}

alphabet = {'a', 'b'}
dfa, start = nfa_to_dfa(nfa, alphabet)

print("DFA states: ")
for state, transitions in dfa.items():
    print(f"  {set(state)} -> { {sym: set(target) for sym, target in transitions.items()} }")

print("\nStart state:", set(start))


DFA states: 
  {'q0'} -> {'b': set(), 'a': {'q1', 'q0'}}
  set() -> {'b': set(), 'a': set()}
  {'q1', 'q0'} -> {'b': {'q2'}, 'a': {'q1', 'q0'}}
  {'q2'} -> {'b': set(), 'a': set()}

Start state: {'q0'}


## Practice exercises

### Exercise 1: easy

Given the NFA:

- states = {q0, q1}
- alphabet = {a}
- start state = q0
- accepting state = q1
- transitions: q0 --a--> q1

Construct the equivalent DFA and explain its states.

### Exercise 2: medium

Consider the NFA:

- states = {q0, q1, q2}
- alphabet = {a, b}
- start = q0
- accepting = q2
- transitions:
  - q0 --a--> q0
  - q0 --a--> q1
  - q1 --b--> q2

Draw the subset-construction states and identify the accepting DFA states.

### Exercise 3: challenging

Create an NFA with epsilon transitions that accepts strings of the form `a* b` or `a* c`. Then convert it to an equivalent DFA and describe how the subset states change as the input is read.

### Exercise 4: implementation challenge

Modify the Python function so that it can handle epsilon transitions explicitly. Test the conversion on an NFA where q0 has an epsilon transition to q1 and q1 has an epsilon transition to q2.

## Final summary

This notebook showed how the NFA-to-DFA conversion works by tracking sets of possible states. The main idea is that a DFA state is really a subset of NFA states. The subset construction is the core reason why NFAs and DFAs recognize the same class of languages.

The most important lesson is this: a DFA can simulate an NFA by storing all possible current NFA states after each input symbol. This converts nondeterminism into a deterministic machine without changing the language recognized.


## Summary

In this notebook, we learned that:

- a DFA is deterministic, but an NFA may have several possible next states,
- an NFA and a DFA can recognize the same language,
- subset construction is the standard method for converting an NFA into an equivalent DFA,
- each DFA state represents a set of NFA states,
- epsilon closure is needed to account for epsilon transitions,
- the conversion is conceptually simple even though the number of DFA states can grow.

The conversion is one of the central ideas in automata theory because it gives a systematic way to turn a flexible machine into a deterministic one without losing expressive power.
